# Replacing the step with the sigmoid

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-4, -2, 0, 2, 4]:
    print(f"sigmoid({z:2d}) = {sigmoid(z):.3f}")

sigmoid(-4) = 0.018
sigmoid(-2) = 0.119
sigmoid( 0) = 0.500
sigmoid( 2) = 0.881
sigmoid( 4) = 0.982


## y - y_hat is never 0 any more

In [2]:
for name, y_true, y_hat in [("A: positive, correct side", 1, 0.8), ("B: negative, correct side", 0, 0.3),
                            ("C: positive, wrong side", 1, 0.3), ("D: negative, wrong side", 0, 0.65)]:
    print(f"{name:28s} y - y_hat = {y_true - y_hat:+.2f}")

A: positive, correct side    y - y_hat = +0.20
B: negative, correct side    y - y_hat = -0.30
C: positive, wrong side      y - y_hat = +0.70
D: negative, wrong side      y - y_hat = -0.65


## Step perceptron vs sigmoid perceptron vs logistic regression

In [3]:
X, y = make_classification(n_samples=100, n_features=2, n_informative=1, n_redundant=0, n_classes=2,
                           n_clusters_per_class=1, random_state=41, hypercube=False, class_sep=30)
Xb = np.insert(X, 0, 1, axis=1)

def perceptron(f, lr=0.1, loops=1000, seed=0):
    rng = np.random.default_rng(seed)
    w = np.ones(3)
    for _ in range(loops):
        j = rng.integers(0, len(y))
        w = w + lr * (y[j] - f(Xb[j] @ w)) * Xb[j]
    return w

def gaps(w):
    d = (Xb @ w) / np.linalg.norm(w[1:])
    return round(-d[y == 0].max(), 2), round(d[y == 1].min(), 2)

step = lambda z: 1.0 if z > 0 else 0.0
lr = LogisticRegression().fit(X, y)
for name, w in [("step", perceptron(step)), ("sigmoid", perceptron(sigmoid)),
                ("logistic regression", np.r_[lr.intercept_, lr.coef_[0]])]:
    print(f"{name:20s} w = {np.round(w, 2)}  gaps (blue, green) = {gaps(w)}")

step                 w = [1.2  1.09 0.52]  gaps (blue, green) = (np.float64(2.41), np.float64(0.25))
sigmoid              w = [2.57 2.28 0.19]  gaps (blue, green) = (np.float64(3.07), np.float64(1.21))
logistic regression  w = [3.55 1.77 0.01]  gaps (blue, green) = (np.float64(2.2), np.float64(1.97))


## Test: more loops, and the same small penalty as scikit-learn
scikit-learn's default `LogisticRegression(C=1)` adds an L2 penalty. Per update, the matching penalty is `w[1:] / (C * n)`.

In [4]:
def perceptron_l2(lr=0.1, loops=1000, lam=0.0, seed=0):
    rng = np.random.default_rng(seed)
    w = np.ones(3)
    for _ in range(loops):
        j = rng.integers(0, len(y))
        w = w + lr * ((y[j] - sigmoid(Xb[j] @ w)) * Xb[j] - lam * np.r_[0, w[1:]])
    return w

for loops in [1000, 100_000, 1_000_000]:
    print(f"{loops:>9} loops  no penalty {gaps(perceptron_l2(loops=loops))}   with penalty {gaps(perceptron_l2(loops=loops, lam=1/len(y)))}")

     1000 loops  no penalty (np.float64(3.07), np.float64(1.21))   with penalty (np.float64(2.64), np.float64(1.57))


   100000 loops  no penalty (np.float64(2.61), np.float64(1.62))   with penalty (np.float64(2.19), np.float64(1.93))


  1000000 loops  no penalty (np.float64(2.48), np.float64(1.76))   with penalty (np.float64(2.18), np.float64(2.0))
